# Delta Lake の挙動を確認する
- [Delta Lake とは何か](https://ktksq.hatenablog.com/entry/deltalake)
- []()

In [0]:
%run ./00_config

In [0]:
# 調査するテーブル名
table_name = "bz_users"

### 1. カタログ・スキーマのデフォルト設定のチェック

In [0]:
spark.sql('SELECT current_catalog(), current_schema()').display()

### 2. テーブル詳細を確認する

In [0]:
display(spark.sql(f'DESCRIBE EXTENDED {table_name};'))

### 3. Deltaテーブルに格納されているファイルを確認（1回目）

In [0]:
# テーブルの場所（Location）を取得
location = spark.sql(f"DESCRIBE DETAIL {table_name}").select("location").collect()[0][0]
print(location)

# 直下の構造（Parquet と _delta_log）を表示
display(dbutils.fs.ls(location))

In [0]:
# _delta_log ディレクトリ配下の構造を表示
delta_log_path = f"{location}/_delta_log"
print(f"Delta Log Path: {delta_log_path}\n")

display(dbutils.fs.ls(delta_log_path))

### 4. Deltaテーブルを更新してみる

In [0]:
# テストデータを1レコード追加
spark.sql(f"""
    INSERT INTO {table_name} 
    VALUES (
        'U99999999',
        'テストユーザー',
        'M',
        '1990-01-01',
        '東京都',
        '090-1234-5678',
        'test@example.com',
        true,
        '2026-01-29',
        'active',
        '2026-01-29',
        null,
        current_timestamp(),
        '/test/data'
    )
""")

print(f"テストデータを {table_name} に追加しました (user_id: U99999999)")

### 5. Deltaテーブルに格納されているファイルを確認（2回目）

In [0]:
# テーブルの場所（Location）を取得
location = spark.sql(f"DESCRIBE DETAIL {table_name}").select("location").collect()[0][0]
print(location)

# 直下の構造（Parquet と _delta_log）を表示
display(dbutils.fs.ls(location))

In [0]:
# _delta_log ディレクトリ配下の構造を表示
delta_log_path = f"{location}/_delta_log"
print(f"Delta Log Path: {delta_log_path}\n")

display(dbutils.fs.ls(delta_log_path))

### おまけ. 対象テーブルがDeltaテーブルか自動チェックする
- [Programmatically determine if a table is a Delta table or not](https://kb.databricks.com/delta/programmatically-determine-if-a-table-is-a-delta-table-or-not)

In [0]:
def delta_check(TableName: str) -> bool:
    desc_table = spark.sql(
        f"describe formatted {TableName}"
    ).collect()
    location = [i[1] for i in desc_table if i[0] == 'Location'][0]
    try:
        dir_check = dbutils.fs.ls(f"{location}/_delta_log")
        is_delta = True
    except Exception:
        is_delta = False
    return is_delta

res = delta_check("komae_demo_v4.medallion_handson.bz_order_items")

if res:
    print("Yes, it is a Delta table.")
else:
    print("No, it is not a Delta table.")